
This notebook uses the fine-tuned detector from Exercise 2 to estimate traffic entering the junction from four directions. It then builds time series and compares ARIMA/SARIMAX forecasting with a naive persistence baseline.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Exercise_3":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPT = PROJECT_ROOT / "Exercise_3" / "exercise3_flow_forecasting.py"
WEIGHTS = PROJECT_ROOT / "Exercise_2" / "runs" / "train" / "yolov8n_zhandong_v2" / "weights" / "best.pt"
RESULTS_DIR = PROJECT_ROOT / "Exercise_3" / "results"
DEVICE = "0"

print("Project:", PROJECT_ROOT)
print("Script:", SCRIPT)
print("Detector weights:", WEIGHTS)
print("Weights exist:", WEIGHTS.exists())

Project: C:\Users\Admin\Desktop\Project AdvanceAi\Project_AdvancedAI
Script: C:\Users\Admin\Desktop\Project AdvanceAi\Project_AdvancedAI\Exercise_3\exercise3_flow_forecasting.py
Detector weights: C:\Users\Admin\Desktop\Project AdvanceAi\Project_AdvancedAI\Exercise_2\runs\train\yolov8n_zhandong_v2\weights\best.pt
Weights exist: True


Run this first. It only draws the four incoming-road counting lines, so it is safe to run before the Exercise 2 model finishes. If the overlay looks wrong, edit `Exercise_3/counting_lines.json`.

In [2]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--overlay-only",
    "--source", "zhandong_road1",
], check=True)

print("Overlay:", RESULTS_DIR / "counting_lines_overlay_zhandong_road1.jpg")

Overlay: C:\Users\Admin\Desktop\Project AdvanceAi\Project_AdvancedAI\Exercise_3\results\counting_lines_overlay_zhandong_road1.jpg


In [3]:
assert WEIGHTS.exists(), "Run Exercise 2 training first; best.pt is missing."

subprocess.run([
    sys.executable, str(SCRIPT),
    "--source", "zhandong_road1",
    "--device", DEVICE,
    "--batch", "16",
    "--imgsz", "640",
    "--conf", "0.25",
], check=True)

CompletedProcess(args=['C:\\Users\\Admin\\anaconda3\\python.exe', 'C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\exercise3_flow_forecasting.py', '--source', 'zhandong_road1', '--device', '0', '--batch', '16', '--imgsz', '640', '--conf', '0.25'], returncode=0)


This produces the same outputs for `zhandong_road2`.

In [4]:
subprocess.run([
    sys.executable, str(SCRIPT),
    "--source", "zhandong_road2",
    "--device", DEVICE,
    "--batch", "16",
    "--imgsz", "640",
    "--conf", "0.25",
], check=True)

CompletedProcess(args=['C:\\Users\\Admin\\anaconda3\\python.exe', 'C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\exercise3_flow_forecasting.py', '--source', 'zhandong_road2', '--device', '0', '--batch', '16', '--imgsz', '640', '--conf', '0.25'], returncode=0)

In [5]:
summary_path = RESULTS_DIR / "summary_zhandong_road1.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print("Summary not created yet. Run the detector/counting cell first.")

{
  "source": "zhandong_road1",
  "images": 831,
  "detections": 81390,
  "total_crossings": {
    "west_in": 144,
    "east_in": 201,
    "north_in": 175,
    "south_in": 115
  },
  "total_flow": 635,
  "bin_seconds": 10,
  "forecast_steps": 360,
  "outputs": {
    "overlay": "C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\results\\counting_lines_overlay_zhandong_road1.jpg",
    "detections": "C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\results\\detections_zhandong_road1.csv",
    "flow_frame_level": "C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\results\\flow_frame_level_zhandong_road1.csv",
    "flow_aggregated": "C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\results\\flow_10s_zhandong_road1.csv",
    "forecast_metrics": "C:\\Users\\Admin\\Desktop\\Project AdvanceAi\\Project_AdvancedAI\\Exercise_3\\results\\forecast_metrics_zhandong_road1.csv"
  }
}
